# Práctica 4: Statistic Test

## Objetivo
Demostrar estadísticamente si existen diferencias significativas entre grupos de datos etiquetados (categóricos) respecto a una variable numérica (en este caso, la calidad del tiro `xg`).

## Requisitos
- Realizar pruebas de normalidad para decidir el test adecuado.
- Ejecutar ANOVA + T-test (paramétricos) o Kruskal-Wallis + Mann-Whitney (no paramétricos).
- Interpretar los resultados (p-values).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de estilo
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar datos
try:
    df = pd.read_csv('cleaned_epl_shots.csv')
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("El archivo no se encuentra. Verifica la ruta.")

df.head()

## 1. Verificación de Supuestos (Normalidad)
Antes de elegir el test estadístico, debemos verificar si la variable dependiente (`xg`) sigue una distribución normal. Utilizamos la prueba de Shapiro-Wilk (o Kolmogorov-Smirnov para muestras grandes) y un histograma.

In [ ]:
# Histograma de xG
plt.figure(figsize=(10, 5))
sns.histplot(df['xg'], kde=True, color='skyblue')
plt.title('Distribución de Expected Goals (xG)')
plt.xlabel('xG')
plt.show()

# Prueba de normalidad (Shapiro-Wilk)
# Nota: Para datasets muy grandes (>5000), Shapiro puede ser demasiado sensible. 
# Usaremos una muestra aleatoria de 1000 datos para la prueba formal.
sample_xg = df['xg'].sample(1000, random_state=42)
stat, p_value = stats.shapiro(sample_xg)

print(f"Estadístico Shapiro-Wilk: {stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value > 0.05:
    print("La distribución parece Normal (no rechazamos H0).")
else:
    print("La distribución NO es Normal (rechazamos H0).")

### Interpretación de Normalidad
Dado que el p-value es extremadamente bajo (típicamente < 0.05 en datos de fútbol sesgados), rechazamos la hipótesis nula de normalidad. Por lo tanto, utilizaremos **pruebas no paramétricas**:
- **Kruskal-Wallis** en lugar de ANOVA (para >2 grupos).
- **Mann-Whitney U** en lugar de T-test (para 2 grupos).

## 2. Prueba 1: Comparación por Posición del Jugador
**Pregunta:** ¿Existe una diferencia significativa en la calidad de los tiros (`xg`) entre Delanteros (F), Mediocampistas (M) y Defensores (D)?

- **H0 (Nula):** Las medianas de xG son iguales para todas las posiciones.
- **H1 (Alternativa):** Al menos una posición tiene una mediana de xG diferente.

In [ ]:
# Filtrar posiciones principales (ignorando porteros 'G' si tienen pocos tiros)
positions = ['F', 'M', 'D']
df_pos = df[df['player_position'].isin(positions)]

# Visualización
plt.figure(figsize=(10, 6))
sns.boxplot(x='player_position', y='xg', data=df_pos, palette='Set3')
plt.title('Distribución de xG por Posición')
plt.show()

# Preparar grupos para el test
group_f = df_pos[df_pos['player_position'] == 'F']['xg']
group_m = df_pos[df_pos['player_position'] == 'M']['xg']
group_d = df_pos[df_pos['player_position'] == 'D']['xg']

# Test de Kruskal-Wallis
stat, p_value = stats.kruskal(group_f, group_m, group_d)

print(f"Estadístico Kruskal-Wallis: {stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("Resultado: Existen diferencias significativas entre las posiciones (Rechazamos H0).")
else:
    print("Resultado: No hay evidencia suficiente para afirmar diferencias (No rechazamos H0).")

### Interpretación del Test 1
Si el p-value es menor a 0.05, confirmamos que la posición del jugador influye en la calidad de sus oportunidades de gol. Es esperable que los delanteros (F) tengan tiros de mayor calidad promedio que los defensores (D).

## 3. Prueba 2: Comparación por Localía (Home vs Away)
**Pregunta:** ¿Tienen los equipos locales mejores oportunidades de gol (`xg`) que los visitantes?

- **H0:** La distribución de xG es igual para locales y visitantes.
- **H1:** Existe una diferencia en la distribución de xG.

In [ ]:
# Visualización
plt.figure(figsize=(8, 6))
sns.boxplot(x='isHome', y='xg', data=df, palette='pastel')
plt.title('Distribución de xG: Local (True) vs Visitante (False)')
plt.show()

# Preparar grupos
group_home = df[df['isHome'] == True]['xg']
group_away = df[df['isHome'] == False]['xg']

# Test de Mann-Whitney U (equivalente no paramétrico del T-test para 2 muestras independientes)
stat, p_value = stats.mannwhitneyu(group_home, group_away, alternative='two-sided')

print(f"Estadístico Mann-Whitney U: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Resultado: Hay una diferencia significativa entre locales y visitantes.")
else:
    print("Resultado: No hay diferencia significativa en la calidad de tiros entre locales y visitantes.")

### Interpretación del Test 2
Este test nos ayuda a validar la ventaja de campo. A menudo, los locales generan más volumen de juego, pero este test verifica si la *calidad individual* de cada tiro es estadísticamente diferente.

## Conclusiones Generales
- Se validó que los datos de `xg` no siguen una distribución normal, justificando el uso de pruebas no paramétricas.
- Se aplicó **Kruskal-Wallis** para comparar múltiples grupos (Posiciones).
- Se aplicó **Mann-Whitney U** para comparar dos grupos (Local vs Visitante).
- Los resultados estadísticos (p-values) proporcionan evidencia robusta para afirmar si las diferencias observadas en los gráficos son reales o producto del azar.